In [23]:
from pathlib import Path
import sys
import pandas as pd
import io
from typing import Callable
import json

# 노트북 기준 상위 폴더를 프로젝트 루트로 가정
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("현재 경로:", Path.cwd())

from llm.openrouter import chat

현재 경로: c:\Users\gayeo\Desktop\Fin-D\compliance_agent\v1_text


In [20]:
def _strip_json_fence(text: str) -> str:
    cleaned = text.strip()
    if cleaned.startswith("```"):
        lines = cleaned.splitlines()
        if lines and lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].startswith("```"):
            lines = lines[:-1]
        cleaned = "\n".join(lines).strip()
    return cleaned

def _extract_json_array_block(text: str) -> str:
    """응답에 설명 문구가 섞여 있어도 JSON 배열 부분만 추출합니다."""
    cleaned = _strip_json_fence(text)
    start = cleaned.find("[")
    if start == -1:
        return cleaned

    depth = 0
    in_string = False
    escape = False

    for i in range(start, len(cleaned)):
        ch = cleaned[i]

        if in_string:
            if escape:
                escape = False
                continue
            if ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
            continue

        if ch == '"':
            in_string = True
        elif ch == "[":
            depth += 1
        elif ch == "]":
            depth -= 1
            if depth == 0:
                return cleaned[start:i + 1]

    # 끝나는 대괄호를 못 찾으면 가능한 범위만 반환
    return cleaned[start:]



In [21]:
def generate_text(
    dataset_name: str,
    topic: str,
    count: int,
    output_csv_path: str,
    system_prompt: str) -> str:
    user_prompt = f"""
Dataset name: {dataset_name}
Topic: {topic}
Sentence Rows: {count}

Requirements:
- 금융 도메인 기반의 마스킹 텍스트 데이터셋을 생성하세요.
- JSON 배열(Array)만 반환하세요. 마크다운 코드펜스 금지.
- 각 원소는 raw_text, masked_text, masked_word 키를 모두 가져야 합니다.
- 모든 값은 문자열로 생성하세요.
""".strip()

    system_prompt = system_prompt.strip()
    data = None
    last_error = None
    last_candidate = ""

    for attempt in range(3):
        retry_hint = ""
        if attempt > 0:
            retry_hint = (
                "\n\n중요: 이전 응답은 JSON 파싱에 실패했습니다. "
                "유효한 JSON 배열만 출력하고, 배열 외 텍스트를 절대 출력하지 마세요."
            )

        raw_response = chat(
            system_prompt=system_prompt,
            user_prompt=user_prompt + retry_hint,
        )
        candidate = _extract_json_array_block(raw_response)
        last_candidate = candidate

        try:
            data = json.loads(candidate)
            break
        except json.JSONDecodeError as exc:
            last_error = exc

    if data is None:
        preview = last_candidate[:300].replace("\n", "\\n")
        raise ValueError(
            f"LLM JSON 파싱 실패: {last_error}. 응답 미리보기: {preview}"
        )

    if not isinstance(data, list):
        raise ValueError("LLM 응답은 JSON 배열(list)이어야 합니다.")

    expected_keys = {"raw_text", "masked_text", "masked_word"}
    normalized_rows = []
    for i, row in enumerate(data[:count]):
        if not isinstance(row, dict):
            raise ValueError(f"{i}번째 항목이 dict가 아닙니다.")

        missing = expected_keys - set(row.keys())
        if missing:
            raise ValueError(f"{i}번째 항목 키 누락: {missing}")

        normalized_rows.append(
            {
                "raw_text": str(row["raw_text"]),
                "masked_text": str(row["masked_text"]),
                "masked_word": str(row["masked_word"]),
            }
        )

    df = pd.DataFrame(normalized_rows, columns=["raw_text", "masked_text", "masked_word"])

    output_path = Path(output_csv_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, index=False, encoding="utf-8-sig")

    return str(output_path)

In [22]:
system_prompt = """
고품질 금융 마스킹 데이터셋을 생성하세요.
반드시 JSON 배열(Array)만 반환하세요. 마크다운 코드펜스 금지.
각 원소는 반드시 raw_text, masked_text, masked_word 키를 포함해야 합니다.
"""

output_file = generate_text(
    dataset_name="금융 고객 상담 마스킹",
    topic="금융 고객 상담에서 개인정보를 마스킹한 텍스트 데이터셋",
    count=10,
    output_csv_path="data/masking_customer_support_v1.csv",
    system_prompt=system_prompt
)

print(f"생성된 파일: {output_file}")
preview_df = pd.read_csv(output_file)
preview_df.head(3)

AttributeError: 'NoneType' object has no attribute 'strip'

In [14]:
import re

# [PERSON_NAME], [CARD_NUMBER], [계좌번호] 형태 모두 허용
PLACEHOLDER_RE = re.compile(r"\[([^\]]+)\]")

def build_entities(raw_text: str, masked_text: str):
    matches = list(PLACEHOLDER_RE.finditer(masked_text))
    if not matches:
        return []

    entities = []
    raw_pos = 0
    mask_pos = 0

    for i, m in enumerate(matches):
        # 현재 placeholder 앞의 일반 텍스트를 raw_text에서 먼저 소비
        prev_literal = masked_text[mask_pos:m.start()]
        if prev_literal:
            idx = raw_text.find(prev_literal, raw_pos)
            if idx == -1:
                return []
            raw_pos = idx + len(prev_literal)

        ent_type = m.group(1)
        next_mask_start = matches[i + 1].start() if i + 1 < len(matches) else len(masked_text)
        next_literal = masked_text[m.end():next_mask_start]

        # 다음 일반 텍스트가 나오기 전까지를 엔티티 span으로 간주
        if next_literal:
            ent_end = raw_text.find(next_literal, raw_pos)
            if ent_end == -1:
                return []
        else:
            ent_end = len(raw_text)

        entities.append(
            {
                "text": raw_text[raw_pos:ent_end],
                "type": ent_type,
                "start": raw_pos,
                "end": ent_end,
            }
        )

        raw_pos = ent_end
        mask_pos = m.end()

    return entities

# preview_df에 entities 컬럼 생성
preview_df["entities"] = preview_df.apply(
    lambda r: str(build_entities(str(r["raw_text"]), str(r["masked_text"]))),
    axis=1,
)

preview_df[["raw_text", "masked_text", "entities"]].head()

,raw_text,masked_text,entities
0,"안녕하세요, 홍길동 고객님. 귀하의 계좌번호 110-234-567890으로 3월 1...","안녕하세요, 홍길동 고객님. 귀하의 계좌번호 [계좌번호]으로 3월 15일 500,0...","[{'text': '110-234-567890', 'type': '계좌번호', 's..."
1,문의하신 상품에 대해 상세 안내 드립니다. 연락 가능한 휴대폰 번호는 010-987...,문의하신 상품에 대해 상세 안내 드립니다. 연락 가능한 휴대폰 번호는 [휴대폰번호]...,"[{'text': '010-9876-5432', 'type': '휴대폰번호', 's..."
